In [10]:
# 基础数据处理
import numpy as np
import pandas as pd

# 机器学习
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import fbeta_score, confusion_matrix

# 忽略警告
import warnings
warnings.filterwarnings('ignore')

In [11]:
# 读取训练集（根据实际路径调整）
train_df = pd.read_csv('./train.csv')

print(f"训练集大小: {len(train_df)}")
train_df.head()

训练集大小: 30000


,objid,ra,dec,modelMag_u,modelMag_g,modelMag_r,modelMag_i,modelMag_z,type
0,1237668504903614866,249.490153,55.151437,23.68451,21.39811,20.67514,19.99474,19.68875,GALAXY
1,1237678905705496902,352.319214,9.039377,25.50177,22.23817,20.35363,-9999.00000,19.04753,GALAXY
2,1237665129620177669,191.973058,35.668576,23.29363,23.13536,21.54516,20.33303,19.59550,GALAXY
3,1237665373868262174,235.781533,22.548614,23.39743,-9999.00000,20.76465,19.92709,-9999.00000,GALAXY
4,1237679322324074909,34.061757,-5.148214,-9999.00000,-9999.00000,20.99208,20.32286,19.83057,GALAXY


In [12]:
feature_cols = ['ra', 'dec', 'modelMag_u', 'modelMag_g', 'modelMag_r', 'modelMag_i', 'modelMag_z']
X_train = train_df[feature_cols].values
y_train = train_df['type'].values

print(f"特征形状: {X_train.shape}")
print(f"标签分布:\n{pd.Series(y_train).value_counts()}")

特征形状: (30000, 7)
标签分布:
GALAXY    21000
STAR       4500
QSO        4500
Name: count, dtype: int64


In [ ]:
print("开始训练 RandomForest ...")
model = RandomForestClassifier(
    n_estimators=1,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

开始训练 RandomForest ...


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [24]:
# 查看训练集上的表现（仅为参考）
y_train_pred = model.predict(X_train)
f2_train = fbeta_score(y_train, y_train_pred, beta=2, average='weighted')
print(f"训练集 F2 分数: {f2_train:.4f}")

训练集 F2 分数: 1.0000


In [25]:
# 读取 A 榜测试数据
A_df = pd.read_csv('./A.csv')

# 提取特征
X_A = A_df[feature_cols].values

# 预测
y_A_pred = model.predict(X_A)

# 构造提交文件
submit_A = pd.DataFrame({
    'objid': A_df['objid'],
    'type': y_A_pred
})

# 保存
submit_A.to_csv('./A_predict.csv', index=False)
print("A_predict.csv 已保存")
submit_A.head()

A_predict.csv 已保存


,objid,type
0,1237662302431347156,QSO
1,1237667781757567228,QSO
2,1237660583909130243,GALAXY
3,1237667212664373457,STAR
4,1237669517444251819,GALAXY


In [26]:
# 读取 B 榜测试数据
B_df = pd.read_csv('./B.csv')

# 提取特征
X_B = B_df[feature_cols].values

# 预测
y_B_pred = model.predict(X_B)

# 构造提交文件
submit_B = pd.DataFrame({
    'objid': B_df['objid'],
    'type': y_B_pred
})

# 保存
submit_B.to_csv('./B_predict.csv', index=False)
print("B_predict.csv 已保存")
submit_B.head()

B_predict.csv 已保存


,objid,type
0,1237662302431347156,QSO
1,1237667781757567228,QSO
2,1237660583909130243,GALAXY
3,1237667212664373457,STAR
4,1237669517444251819,GALAXY


In [27]:
# 计算 A_predict.csv 和 A_ground_truth.csv 的 F2 分数
A_predict = pd.read_csv('./A_predict.csv')
A_ground_truth = pd.read_csv('./A_ground_truth.csv')

# 计算 F2 分数
f2_A = fbeta_score(A_ground_truth['type'], A_predict['type'], beta=2, average='weighted')
print(f"A_predict.csv 和 A_ground_truth.csv 的 F2 分数: {f2_A:.6f}")

A_predict.csv 和 A_ground_truth.csv 的 F2 分数: 0.760944
